# steam_indie_reviews 수집 현황 확인

In [6]:
import pandas as pd
from datetime import timedelta
from pathlib import Path

ROOT         = Path("../../..").resolve()
REVIEWS_PATH = ROOT / "data/processed/steam_indie_reviews.csv"
SAMPLE_PATH  = ROOT / "data/processed/steam_stratified_sample.csv"
EARLY_DAYS   = 90

sample  = pd.read_csv(SAMPLE_PATH)
reviews = pd.read_csv(REVIEWS_PATH)

sample['release_date'] = pd.to_datetime(sample['release_date'], errors='coerce')
targets = sample[sample['stratum'].isin(['large_high', 'mid_high', 'small_high'])].copy()

print(f"수집 대상 게임: {len(targets)}개")
print(f"수집된 리뷰 총계: {len(reviews):,}건")

수집 대상 게임: 74개
수집된 리뷰 총계: 150,285건


## 1. 게임별 수집 현황

In [7]:
review_counts = (
    reviews.groupby('appid')
    .agg(
        collected=('recommendationid', 'count'),
        ea_reviews=('written_during_early_access', 'sum'),
        min_date=('timestamp_created', lambda x: pd.to_datetime(x.min(), unit='s').date()),
        max_date=('timestamp_created', lambda x: pd.to_datetime(x.max(), unit='s').date()),
    )
    .reset_index()
)

status = targets[['appid', 'name_store', 'stratum', 'release_date', 'total_reviews']].merge(
    review_counts, on='appid', how='left'
)
status['collected']  = status['collected'].fillna(0).astype(int)
status['ea_reviews'] = status['ea_reviews'].fillna(0).astype(int)
status['수집완료']    = status['collected'] > 0
status['cutoff_date'] = status['release_date'] + timedelta(days=EARLY_DAYS)

print(f"수집 완료: {status['수집완료'].sum()}개 / {len(status)}개")
print(f"미수집:    {(~status['수집완료']).sum()}개")
print()
status[['name_store', 'stratum', 'release_date', 'cutoff_date',
        'total_reviews', 'collected', 'ea_reviews', 'min_date', 'max_date']]

수집 완료: 72개 / 74개
미수집:    2개



,name_store,stratum,release_date,cutoff_date,total_reviews,collected,ea_reviews,min_date,max_date
0,Sun Haven,large_high,2023-03-10,2023-06-08,22456,4078,5,2023-03-10,2023-06-07
1,(the) Gnorp Apologue,large_high,2023-12-14,2024-03-13,8143,3438,0,2023-12-14,2024-03-12
2,轮回修仙路,large_high,2023-06-19,2023-09-17,2865,413,7,2023-06-19,2023-09-16
3,MiSide,large_high,2024-12-10,2025-03-10,111087,18695,0,2025-01-20,2025-03-09
4,Necesse,large_high,2025-10-16,2026-01-14,17071,6982,21,2025-10-16,2026-01-13
...,...,...,...,...,...,...,...,...,...
69,NightClub Simulator,small_high,2024-02-26,2024-05-26,453,69,69,2024-02-26,2024-05-19
70,XiuzhenWorld,small_high,2024-04-30,2024-07-29,435,411,0,2024-04-30,2024-07-28
71,东方雪莲华 ～ Abyss Soul Lotus.,small_high,2023-02-02,2023-05-03,427,293,0,2023-02-02,2023-04-29
72,Coin Pusher Casino,small_high,2024-02-29,2024-05-29,731,52,0,2024-02-29,2024-05-28


## 2. 층별 수집 현황

In [8]:
stratum_summary = (
    status.groupby('stratum')
    .agg(
        게임수=('appid', 'count'),
        수집완료=('수집완료', 'sum'),
        총리뷰수=('collected', 'sum'),
        EA리뷰수=('ea_reviews', 'sum'),
        중앙값=('collected', 'median'),
        최솟값=('collected', 'min'),
        최댓값=('collected', 'max'),
    )
)
stratum_summary['미수집'] = stratum_summary['게임수'] - stratum_summary['수집완료']
print(stratum_summary)

            게임수  수집완료    총리뷰수  EA리뷰수     중앙값  최솟값    최댓값  미수집
stratum                                                      
large_high   29    27  131756   7799  2380.0    0  25627    2
mid_high     25    25   13742   2324   297.0    3   2789    0
small_high   20    20    4787    881   257.5    6    625    0


## 3. 미수집 게임 목록

In [9]:
missing = status[~status['수집완료']][['appid', 'name_store', 'stratum', 'release_date', 'total_reviews']]
if missing.empty:
    print("미수집 게임 없음")
else:
    print(f"미수집 게임 {len(missing)}개:")
    print(missing.to_string(index=False))

미수집 게임 2개:
  appid               name_store    stratum release_date  total_reviews
2334220 Home Sweet Home : Online large_high   2023-06-21           2004
2379780                  Balatro large_high   2024-02-20         153566
